In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # type: ignore
from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import category_encoders as ce
import joblib

sns.set(style="whitegrid")
RANDOM_STATE = 42

In [3]:
df = pd.read_csv('../data/raw/DATASETSAMPLE.csv', low_memory=False)
df.head()

,Scan Time,Number Of Legs,Electronic Ticket Indicator,PNRCode,From Airport,To Airport,Carrier Designator,Flight Number,Julian Flight Date,Flight Date,...,Passenger Status,Validated Leg,AOSFlight ID,Scheduled Departure,Estimated Departure,Access Gate Name,Access Gate Location,Action,Delog Reason,Result
0,12/01/2026 02:57,1.0,,TTILXS,STN,CIA,FR,2455.0,12,12/01/2026 00:00,...,1,1,FR2455,12/01/2026 07:00,12/01/2026 07:00,STNSAAG25,T1,Validate,NaN,OK;
1,12/01/2026 02:57,1.0,,ZU9F2Z,STN,VCE,FR,792.0,12,12/01/2026 00:00,...,1,1,FR792,12/01/2026 06:55,12/01/2026 06:55,STNSAAG26,T1,Validate,NaN,OK;
2,12/01/2026 02:58,1.0,,PUC76Z,STN,VIE,FR,7357.0,12,12/01/2026 00:00,...,1,1,FR7357,12/01/2026 06:55,12/01/2026 06:55,STNSAAG24,T1,Validate,NaN,OK;
3,12/01/2026 02:58,1.0,,JTQL4Z,STN,RIX,FR,2642.0,12,12/01/2026 00:00,...,1,1,FR2642,12/01/2026 07:35,12/01/2026 07:35,STNSAAG26,T1,Validate,NaN,OK;
4,12/01/2026 02:58,1.0,,O7ZCVE,STN,VIE,FR,7357.0,12,12/01/2026 00:00,...,1,1,FR7357,12/01/2026 06:55,12/01/2026 06:55,STNSAAG24,T1,Validate,NaN,OK;


In [5]:
df['Result'].value_counts(dropna=False)

Result
OK;                                                  958568
BPBC successfully delogged;                           31488
Duplicate Barcode;                                    27949
Wrong Airport;                                        13701
Invalid IATA format;                                   5050
Wrong Date;                                            2311
Unrecognised Barcode;                                   966
Too Late;                                               855
Grant Access Request (validation not performed);        612
NaN                                                     167
Flight Not found;                                       116
Flight Cancelled;                                        69
Passenger Not Checked In;                                18
Barcode Not Accepted At Gate;                             6
Name: count, dtype: int64

In [6]:
# Clean Result column
df['Result'] = df['Result'].astype(str).str.strip()

# Define missed-flight column label
df['target'] = (df['Result'] == 'Too Late;').astype(int)

df['target'].value_counts()

target
0    1041021
1        855
Name: count, dtype: int64

In [7]:
df['Scan Time'] = pd.to_datetime(df['Scan Time'], dayfirst=True, errors='coerce')
df['Scheduled Departure'] = pd.to_datetime(df['Scheduled Departure'], dayfirst=True, errors='coerce')
df['Estimated Departure'] = pd.to_datetime(df['Estimated Departure'], dayfirst=True, errors='coerce')

In [8]:
df['scan_hour'] = df['Scan Time'].dt.hour.fillna(-1).astype(int)
df['scan_dow'] = df['Scan Time'].dt.dayofweek.fillna(-1).astype(int)
df['scheduled_hour'] = df['Scheduled Departure'].dt.hour.fillna(-1).astype(int)

df['departure_delay_min'] = (
    (df['Estimated Departure'] - df['Scheduled Departure'])
    .dt.total_seconds() / 60
).fillna(0)

In [9]:
high_card_cols = ['Access Gate Name', 'To Airport']
te = ce.TargetEncoder(cols=high_card_cols)
df[high_card_cols] = te.fit_transform(df[high_card_cols], df['target'])

In [10]:
df['airline'] = df['Flight Number'].astype(str).str.extract(r'([A-Za-z]{2,3})', expand=False).fillna('UNK')
df = pd.get_dummies(df, columns=['airline'], drop_first=True)

In [ ]:
# Stratified Train/Test split
# Good for imbalanced classification
from sklearn.model_selection import train_test_split

features = [
    'scan_hour', 'scan_dow', 'scheduled_hour', 'departure_delay_min',
    'Access Gate Name', 'To Airport'
] + [col for col in df.columns if col.startswith('airline_')]

X = df[features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [12]:
X_train.shape, X_test.shape

((833500, 6), (208376, 6))

In [13]:
y_train.value_counts(), y_test.value_counts()

(target
 0    832816
 1       684
 Name: count, dtype: int64,
 target
 0    208205
 1       171
 Name: count, dtype: int64)

In [ ]:
# Scale numeric features 
num_cols = ['scan_hour', 'scan_dow', 'scheduled_hour', 'departure_delay_min',
            'Access Gate Name', 'To Airport']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [ ]:
# Good baseline and interpretability
# Establishes a benchmark
logreg = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)

print("Logistic Regression Report:")
print(classification_report(y_test, y_pred_lr))

c:\Users\Mariasophia Lacsina\Desktop\missed_flight_project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Logistic Regression Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95    208205
           1       0.01      0.83      0.01       171

    accuracy                           0.90    208376
   macro avg       0.50      0.87      0.48    208376
weighted avg       1.00      0.90      0.95    208376



In [ ]:
# Good for stakeholder explanation (?)

dt = DecisionTreeClassifier(
    max_depth=4,
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Report:")
print(classification_report(y_test, y_pred_dt))

Decision Tree Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98    208205
           1       0.01      0.65      0.02       171

    accuracy                           0.95    208376
   macro avg       0.51      0.80      0.50    208376
weighted avg       1.00      0.95      0.97    208376



In [ ]:
# Handles non-linear interactions
# Handles imbalance well
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [6, 12, None],
    'min_samples_split': [2, 6],
    'max_features': ['sqrt', 0.5]
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

grid = GridSearchCV(
    rf,
    param_grid,
    scoring='recall',
    cv=3,
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_rf = grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)

print("Best Params:", grid.best_params_)
print("Random Forest Report:")
print(classification_report(y_test, y_pred_rf))

Best Params: {'max_depth': 6, 'max_features': 0.5, 'min_samples_split': 2, 'n_estimators': 200}
Random Forest Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    208205
           1       0.02      0.85      0.03       171

    accuracy                           0.96    208376
   macro avg       0.51      0.91      0.51    208376
weighted avg       1.00      0.96      0.98    208376



In [1]:
from pathlib import Path
from datetime import datetime
import json

# Paths and versioning
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_filename = f'pipeline_rf_v1_{timestamp}.joblib'
meta_filename = f'metadata_rf_v1_{timestamp}.json'

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import category_encoders as ce
from sklearn.ensemble import RandomForestClassifier

# Lists you used earlier
num_cols = ['scan_hour', 'scan_dow', 'scheduled_hour', 'departure_delay_min']
high_card_cols = ['Access Gate Name', 'To Airport']
# If you prefer to one-hot encode airline instead of precomputing dummies:
airline_col = ['airline']  # only if you have a single airline column

# Define transformers
num_transformer = Pipeline([
    ('scaler', StandardScaler())
])

# TargetEncoder works as an sklearn transformer and can be used in ColumnTransformer
cat_target_transformer = ce.TargetEncoder(cols=high_card_cols)

# If you want to one-hot encode airline inside pipeline
cat_ohe_transformer = Pipeline([
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('gate_enc', cat_target_transformer, high_card_cols),
        ('airline_ohe', cat_ohe_transformer, airline_col)  # remove if airline already dummified
    ],
    remainder='passthrough'  # keep any other columns (e.g., precomputed airline dummies)
)

# Final pipeline with the tuned RandomForest parameters you found
rf_params = {'n_estimators': 200, 'max_depth': 6, 'min_samples_split': 2, 'max_features': 0.5}
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1, **rf_params))
])

In [5]:
# Fit pipeline on training data
pipeline.fit(X_train, y_train)

NameError: name 'X_train' is not defined

In [ ]:
import joblib

# Save pipeline
pipeline_path = models_dir / model_filename
joblib.dump(pipeline, pipeline_path, compress=3)

# Save metadata
metadata = {
    "model_file": str(pipeline_path.name),
    "timestamp": timestamp,
    "model_type": "RandomForestClassifier",
    "rf_params": rf_params,
    "num_cols": num_cols,
    "high_card_cols": high_card_cols,
    "airline_col": airline_col,
    "features_used": features,  # your features list from earlier
    "random_state": 42
}
meta_path = models_dir / meta_filename
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

In [ ]:
# Load and smoke test
loaded_pipeline = joblib.load(pipeline_path)

# Use a small sample from the test set
sample_X = X_test.sample(200, random_state=42)
sample_y = y_test.loc[sample_X.index]

y_pred = loaded_pipeline.predict(sample_X)
from sklearn.metrics import classification_report
print(classification_report(sample_y, y_pred))

In [ ]:
# Later, in a new script or notebook
import joblib
from pathlib import Path

models_dir = Path('../models')
pipeline = joblib.load(models_dir / 'pipeline_rf_v1_20260224_123456.joblib')  # replace with actual filename

# X_new must contain the raw columns expected by the preprocessor:
# num_cols, high_card_cols, airline (if used)
X_new_prepared = X_new.copy()  # ensure column names match exactly
y_proba = pipeline.predict_proba(X_new_prepared)[:, 1]
y_pred = pipeline.predict(X_new_prepared)